In [2]:
import pandas as pd

In [3]:
future_df = pd.read_csv("/Users/rhombus19/projects/eleo/future_features_dataset.csv")

In [4]:
train_df = pd.read_csv("/Users/rhombus19/projects/eleo/forecast_dataset.csv")

In [5]:
future_df["date"] = pd.to_datetime(future_df["date"])
train_df["date"] = pd.to_datetime(train_df["date"])

In [6]:
train_df.sort_values(by="date", inplace=True)
train_df.reset_index(drop=True, inplace=True)

In [7]:
import pandas as pd
from pytorch_forecasting import TimeSeriesDataSet

# make sure date is datetime
train_df["date"] = pd.to_datetime(train_df["date"])

# sort by group + time (recommended)
train_df = train_df.sort_values(["SKU", "date"])

# create integer time index: days since first date
train_df["time_idx"] = (train_df["date"] - train_df["date"].min()).dt.days.astype("int64")


In [8]:
train_df["SKU"] = train_df["SKU"].astype("category")

# make calendar features string-based, then categorical
train_df["month"] = train_df["month"].astype(str).astype("category")
train_df["day_of_week"] = train_df["day_of_week"].astype(str).astype("category")

# bool is treated as numeric -> convert to string labels
train_df["is_holiday"] = train_df["is_holiday"].map(
    {True: "holiday", False: "no_holiday"}
).astype("category")

In [11]:
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting import TemporalFusionTransformer
from pytorch_forecasting.metrics import QuantileLoss

# specify normalizer to handle zero values correctly
normalizer = GroupNormalizer(groups=["SKU"], transformation="softplus")

tft_dataset = TimeSeriesDataSet(
    train_df,
    time_idx="time_idx",
    target="qty",
    group_ids=["SKU"],
    static_categoricals=["SKU"],
    time_varying_known_reals=["price_per_unit", "tavg", "prcp", "tsun", "sale_percent"],
    time_varying_known_categoricals=["month", "day_of_week", "is_holiday"],
    time_varying_unknown_reals=["qty"],
    max_encoder_length=365,
    max_prediction_length=30,
    target_normalizer=normalizer,
)


In [12]:
from pytorch_forecasting.metrics import PoissonLoss
import lightning.pytorch as pl

train_dataloader = tft_dataset.to_dataloader(train=True, batch_size=64)

model = TemporalFusionTransformer.from_dataset(
    tft_dataset,
    learning_rate=1e-3,
    hidden_size=64,
    attention_head_size=4,
    dropout=0.1,
    loss=PoissonLoss(),  # or QuantileLoss()
    log_interval=10,
    reduce_on_plateau_patience=4,
)

trainer = pl.Trainer(max_epochs=30, accelerator="auto")
trainer.fit(model, train_dataloaders=train_dataloader)


/Users/rhombus19/projects/eleo/eleo-mind/packages/backend/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:210: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/Users/rhombus19/projects/eleo/eleo-mind/packages/backend/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:210: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
/Users/rhombus19/projects/eleo/eleo-mind/packages/backend/.venv/lib/py

Training: |          | 0/? [00:00<?, ?it/s]


Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1

/Users/rhombus19/projects/eleo/eleo-mind/packages/backend/.venv/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
